# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DjebrilSVN/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** One row = one content item (page) per client.
**Time window:** The single month of March 2026 (`month=2026-03`).


In [ ]:
# This cell is for CODE (numbers, a query, a check).
import duckdb
import os
import sys

con = duckdb.connect()

token = os.environ.get('HF_TOKEN')
if not token and "google.colab" in sys.modules:
    from google.colab import userdata
    try:
        token = userdata.get('HF_TOKEN')
    except userdata.SecretNotFoundError:
        print("HF_TOKEN secret not found in Colab.")

if token:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token}')")
else:
    print("Warning: No HF_TOKEN found. Gated queries will fail.")

REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"


## 2. Fields: feature / label / context / excluded

- **Feature:** `impressions`, `clicks`, `sessions`, `engaged_sessions`, and `gsc_avg_position` aggregated over the time window. Knowable at the decision moment because they are historical totals.
- **Label / proxy:** `is_high_traffic` (a proxy indicator if total impressions > 1000). Since my original lane was clustering, I am predicting this proxy label here to demonstrate the leakage trap.
- **Context:** `client_id`, `content_id`. Used only to identify and group pages, never as features.
- **Excluded:** `report_date` (aggregated away). We also deliberately exclude any rows where `ga4_data_available` or `gsc_data_available` is NOT TRUE, because missing data periods would incorrectly masquerade as zero engagement.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

We will run three queries to prove the grain, row count & date span, and availability.


In [ ]:
# 1. Grain: Does one row equal one report_date x client x content in the raw table?
grain_check = con.sql(f"""
SELECT report_date, client_id, content_id, COUNT(*) c
FROM {REL}
GROUP BY report_date, client_id, content_id
HAVING c > 1 LIMIT 5
""").df()
print(f"Grain violations: {len(grain_check)} rows")

# 2. Row count and date span
count_span = con.sql(f"""
SELECT COUNT(*) as row_count, MIN(report_date) as start_date, MAX(report_date) as end_date
FROM {REL}
""").df()
print("\nRow count and date span:")
print(count_span)

# 3. Availability filter (IS TRUE)
avail = con.sql(f"""
SELECT COUNT(*) as surviving_rows
FROM {REL}
WHERE ga4_data_available IS TRUE AND gsc_data_available IS TRUE
""").df()
print("\nSurviving rows after availability filter:")
print(avail)


## 4. Five features and the Leakage Trap

We build a 5-feature frame. 
- `impressions`: Knowable at the decision moment because it's a past total.
- `sessions`: Knowable at the decision moment because it's a past total.
- `engaged_sessions`: Knowable at the decision moment because it's a past total.
- `avg_position`: Knowable at the decision moment because it's a past average.
- `ctr`: Knowable at the decision moment because it's derived from past clicks and impressions.

**The Trap:** We create a proxy label `is_high_traffic = (impressions > 1000)`. Then we deliberately add `clicks` (a label-derived column in this context, since high clicks guarantee high impressions) to the features. The score will jump toward perfect. Then we remove it to see the honest score.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Build the feature frame
feature_frame = con.sql(f"""
SELECT 
    client_id, 
    content_id, 
    SUM(impressions) as impressions,
    SUM(clicks) as clicks,  -- The Trap feature
    SUM(sessions) as sessions,
    SUM(engaged_sessions) as engaged_sessions,
    AVG(gsc_avg_position) as avg_position
FROM {REL}
WHERE ga4_data_available IS TRUE AND gsc_data_available IS TRUE
GROUP BY client_id, content_id
HAVING SUM(impressions) > 0  -- Filter out zero-impression rows for CTR calculation
""").df()

# Calculate CTR and Proxy Label
feature_frame['ctr'] = (feature_frame['clicks'] / feature_frame['impressions']) * 100
feature_frame['is_high_traffic'] = (feature_frame['impressions'] > 1000).astype(int)

# Define features (including the trap)
features_with_trap = ['sessions', 'engaged_sessions', 'avg_position', 'ctr', 'clicks']
features_honest = ['sessions', 'engaged_sessions', 'avg_position', 'ctr']

X_trap = feature_frame[features_with_trap].fillna(0)
X_honest = feature_frame[features_honest].fillna(0)
y = feature_frame['is_high_traffic']

if len(feature_frame) > 0:
    X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(X_trap, y, test_size=0.2, random_state=42)
    clf_trap = LogisticRegression(max_iter=1000).fit(X_train_t, y_train_t)
    trap_score = accuracy_score(y_test_t, clf_trap.predict(X_test_t))
    print(f"Score WITH leakage trap ('clicks' included): {trap_score:.3f}")
    
    X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_honest, y, test_size=0.2, random_state=42)
    clf_honest = LogisticRegression(max_iter=1000).fit(X_train_h, y_train_h)
    honest_score = accuracy_score(y_test_h, clf_honest.predict(X_test_h))
    print(f"Honest Score (without 'clicks'): {honest_score:.3f}")


## 5. Data limits

**One limitation:** The panel is unbalanced. Rows before a client's `ga4_data_start` have GA4 columns zero-filled or NULL. If we don't explicitly filter by `ga4_data_available IS TRUE`, we might misinterpret missing data periods as "zero engagement" periods, drastically skewing aggregations.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.